In [1]:
import os
import random
import re

import numpy as np
import pandas as pd
import tensorflow as tf

# Fix for TF/Keras seeding in TF 2.x (set_random_seed is not available)
seed = 12345
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Use the Kaggle-provided paths from the prompt
BASE_INPUT = "/kaggle/input/movie-review-sentiment-analysis-kernels-only"
if not os.path.exists(BASE_INPUT):
    # Fallback to generic /kaggle/input if dataset is mounted differently
    BASE_INPUT = "/kaggle/input"

print("BASE_INPUT:", BASE_INPUT)
print("Input dir listing (truncated):", os.listdir(BASE_INPUT)[:20])



2026-01-13 19:02:43.966127: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768330963.978580      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768330963.982466      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-13 19:02:43.998382: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

BASE_INPUT: /kaggle/input/movie-review-sentiment-analysis-kernels-only
Input dir listing (truncated): ['train.tsv.zip', 'movie-review-sentiment-analysis-kernels-only', 'sampleSubmission.csv', 'test.tsv.zip', 'description.md', 'sampleSubmission.csv.zip', 'test.tsv', 'train.tsv']


In [2]:
# Read data (tsv for train/test, csv for sample submission)
train_path = os.path.join(BASE_INPUT, "train.tsv")
test_path = os.path.join(BASE_INPUT, "test.tsv")
sample_path = os.path.join(BASE_INPUT, "sampleSubmission.csv")

train = pd.read_csv(train_path, sep="\t")
test = pd.read_csv(test_path, sep="\t")
sampleSubmission = pd.read_csv(sample_path)

print(train.shape, test.shape, sampleSubmission.shape)
train.head(3)




(109242, 4) (46818, 3) (46818, 2)


,PhraseId,SentenceId,Phrase,Sentiment
0,24144,1097,through every frame,2
1,85111,4402,onscreen presence,2
2,75999,3900,into dysfunction,2


In [3]:
def clean_str(string):
    """
    Tokenization/string cleaning for all datasets except for SST.
    """
    string = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", string)
    string = re.sub(r"\'s", " 's", string)
    string = re.sub(r"\'ve", " 've", string)
    string = re.sub(r"n\'t", " n't", string)
    string = re.sub(r"\'re", " 're", string)
    string = re.sub(r"\'d", " 'd", string)
    string = re.sub(r"\'ll", " 'll", string)
    string = re.sub(r",", " , ", string)
    string = re.sub(r"!", " ! ", string)
    string = re.sub(r"\(", " \( ", string)
    string = re.sub(r"\)", " \) ", string)
    string = re.sub(r"\?", " \? ", string)
    string = re.sub(r"\s{2,}", " ", string)
    return string.strip().lower()


phrases = [clean_str(s) for s in train["Phrase"].astype(str).values]



In [4]:
# Keep same label encoding logic; use tf.keras equivalent for compatibility
from tensorflow.keras.utils import to_categorical

Y = to_categorical(train["Sentiment"].values, num_classes=5)
print(Y[155:165])
print(train["Sentiment"].values[155:165])



[[0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]]
[2 2 2 2 3 4 3 3 2 2]


In [5]:
# Fix: keras.preprocessing.* removed in Keras 3; use tf.keras.preprocessing instead
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

t = Tokenizer()
t.fit_on_texts(phrases)
vocab_size = len(t.word_index) + 1

X = t.texts_to_sequences(phrases)
max_length = max(len(p.split()) for p in phrases)

X = pad_sequences(X, maxlen=max_length, padding="post")
print("X shape:", X.shape, "vocab_size:", vocab_size, "max_length:", max_length)



X shape: (109242, 52) vocab_size: 15262 max_length: 52


In [6]:
# Fix: invalid import "merge" and Keras 3 API issues; keep same model3 core logic
from tensorflow.keras.layers import (
    Embedding,
    Dense,
    Flatten,
    Conv1D,
    SpatialDropout1D,
    MaxPooling1D,
    Input,
    Dropout,
    concatenate,
)
from tensorflow.keras.models import Sequential, Model


def model1(output_dim=8, max_length=50, y_dim=5, filter_sizes=[3]):
    model = Sequential()
    model.add(Embedding(vocab_size, output_dim, input_length=max_length))
    model.add(Flatten())
    model.add(Dense(y_dim, activation="sigmoid"))

    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["acc"])
    print(model.summary())
    return model


def model2(output_dim=8, max_length=50, y_dim=5, num_filters=5, filter_sizes=[3, 5]):
    model = Sequential()
    model.add(Embedding(vocab_size, output_dim, input_length=max_length))
    model.add(SpatialDropout1D(0.2))
    model.add(
        Conv1D(
            num_filters, kernel_size=filter_sizes[0], padding="valid", activation="relu"
        )
    )
    model.add(
        MaxPooling1D(
            pool_size=max_length - filter_sizes[0] + 1, strides=1, padding="valid"
        )
    )
    model.add(Flatten())
    model.add(Dense(y_dim, activation="sigmoid"))

    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["acc"])
    print(model.summary())
    return model


def model3(output_dim=8, max_length=50, y_dim=5, num_filters=5, filter_sizes=[3, 5]):
    embed_input = Input(shape=(max_length,))
    x = Embedding(vocab_size, output_dim, input_length=max_length)(embed_input)
    pooled_outputs = []
    for fs in filter_sizes:
        conv = Conv1D(num_filters, kernel_size=fs, padding="valid", activation="relu")(
            x
        )
        conv = MaxPooling1D(pool_size=max_length - fs + 1, strides=1, padding="valid")(
            conv
        )
        pooled_outputs.append(conv)
    merged = concatenate(pooled_outputs)

    x = Flatten()(merged)
    x = Dropout(0.2)(x)
    predictions = Dense(y_dim, activation="sigmoid")(x)

    model = Model(inputs=embed_input, outputs=predictions)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["acc"])
    print(model.summary())
    return model


model = model3(output_dim=8, max_length=max_length, y_dim=5, filter_sizes=[3, 4, 5])



/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-01-13 19:02:52.885171: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1768330972.885561      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 40007 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:45:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 52)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 52, 8)     │    122,096 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 50, 5)     │        125 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 49, 5)     │        165 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 48, 5)     │        205 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 1, 5)      │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 1, 5)      │          0 │ conv1d_1[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 1, 5)      │          0 │ conv1d_2[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1, 15)     │          0 │ max_pooling1d[0]… │
│ (Concatenate)       │                   │            │ max_pooling1d_1[… │
│                     │                   │            │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 15)        │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 15)        │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 5)         │         80 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 122,671 (479.18 KB)

 Trainable params: 122,671 (479.18 KB)

 Non-trainable params: 0 (0.00 B)

None


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.25, random_state=seed
)
print(X_train.shape, X_val.shape, Y_train.shape, Y_val.shape)



(81931, 52) (27311, 52) (81931, 5) (27311, 5)


In [8]:
epochs = 20
batch_size = 32

model.fit(
    X_train,
    Y_train,
    epochs=epochs,
    validation_data=(X_val, Y_val),
    batch_size=batch_size,
    verbose=1,
)
loss, accuracy = model.evaluate(X_val, Y_val, verbose=0)
print("Accuracy: %f" % (accuracy * 100))



Epoch 1/20


I0000 00:00:1768330974.302629      62 service.cc:148] XLA service 0x7f67e800b0d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768330974.302661      62 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-13 19:02:54.325977: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1768330974.433935      62 cuda_dnn.cc:529] Loaded cuDNN version 90300


 120/2561 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - acc: 0.5126 - loss: 1.4853

I0000 00:00:1768330976.051905      62 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2561/2561 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - acc: 0.5196 - loss: 1.2749 - val_acc: 0.5834 - val_loss: 1.0374
Epoch 2/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - acc: 0.6082 - loss: 1.0017 - val_acc: 0.6240 - val_loss: 0.9302
Epoch 3/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.6506 - loss: 0.8763 - val_acc: 0.6336 - val_loss: 0.9132
Epoch 4/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.6739 - loss: 0.8147 - val_acc: 0.6381 - val_loss: 0.9147
Epoch 5/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.6884 - loss: 0.7775 - val_acc: 0.6410 - val_loss: 0.9218
Epoch 6/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.6978 - loss: 0.7448 - val_acc: 0.6427 - val_loss: 0.9338
Epoch 7/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.7075 - loss: 0.7270 - val_acc: 0.6437 - val_loss: 0.9444
Epoch 8/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.7159 - loss: 0.7041 - val_acc: 0.6438 - val_loss: 0.9605
Epoch 9/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/st

In [9]:
test_phrases = [clean_str(s) for s in test["Phrase"].astype(str).values]
X_test = t.texts_to_sequences(test_phrases)
X_test = pad_sequences(X_test, maxlen=max_length, padding="post")
print("X_test shape:", X_test.shape)



X_test shape: (46818, 52)


In [10]:
# Ensure correct submission columns and alignment with PhraseId
pred = model.predict(X_test, verbose=1).argmax(axis=-1)

# Use provided sampleSubmission as template to preserve correct PhraseId order/count
sampleSubmission["Sentiment"] = pred.astype(int)

out_path = "sub_cnn3.csv"
sampleSubmission.to_csv(out_path, index=False)

print("Wrote:", out_path)
print(sampleSubmission.head())
print(sampleSubmission.shape)

1464/1464 ━━━━━━━━━━━━━━━━━━━━ 1s 686us/step
Wrote: sub_cnn3.csv
   PhraseId  Sentiment
0     21924          2
1     18207          4
2     81803          2
3    114983          3
4    135928          3
(46818, 2)
